In [0]:

-- Quick data check

SELECT *
FROM walmart_data;

SELECT COUNT(*) AS total_count
FROM walmart_data;

SELECT DISTINCT payment_method, COUNT(*)
FROM walmart_data
GROUP BY payment_method;

SELECT COUNT(DISTINCT branch)
FROM walmart_data;

-- Business Problems

-- 1. Find different payment method and number of transactions, number of qty sold
SELECT payment_method, COUNT(*) AS total_transactions, SUM(quantity) AS quantity_sold
FROM walmart_data
GROUP BY payment_method;

-- 2. Identity the highest-rated category in each branch, displaying the branch, category
-- AVG Rating

SELECT *
FROM (
    SELECT
        branch,
        category,
        AVG(rating) AS avg_rating,
        RANK() OVER (
            PARTITION BY branch
            ORDER BY AVG(rating) DESC
        ) AS rank
    FROM walmart_data
    GROUP BY branch, category
)
WHERE rank = 1;

-- 3. Identity the busiest day for each branch based on the number of transactions
SELECT *
FROM (
    SELECT 
        COUNT(*) AS no_transactions, 
        branch, 
        DATE_FORMAT(date, 'EEEE') AS day_name,
        RANK() OVER (
                PARTITION BY branch
                ORDER BY COUNT(*) DESC
            ) AS rank
        FROM walmart_data
    GROUP BY branch, DATE_FORMAT(date, 'EEEE')
)
WHERE rank = 1;

-- 4. Calculate the total quantity of items sold per payment method. List payment_method and total_quantity.
SELECT payment_method, SUM(quantity) AS total_quantity
FROM walmart_data
GROUP BY payment_method;

-- 5. Determine the average, minimum, and maximum rating of category for each city
SELECT city, category, AVG(rating), MIN(rating), MAX(rating)
FROM walmart_data
GROUP BY city, category
ORDER by city, category;

-- 6. Calculate the total profit for each category by considering total_profit as
-- (unit_price * quantity * profit_margin).
-- List category and total_profit, ordered from highest to lowest profit.
SELECT branch, SUM(unit_price * quantity * profit_margin) AS total_profit, category
FROM walmart_data
GROUP BY Branch, category
ORDER BY Branch, total_profit DESC;

-- 7. Determine the most common payment method for each Branch.
-- Display Branch and the preferred_payment_method.
SELECT payment_method, COUNT(*) AS no_transactions, Branch
FROM walmart_data
GROUP BY payment_method, Branch
ORDER BY Branch, no_transactions DESC;

-- 8. Categorize sales into 3 group MORNING, AFTERNOON, EVENING
-- Find out each of the shift and number of invoices
SELECT branch,
CASE 
    WHEN EXTRACT(HOUR FROM(time::time)) < 12 THEN "Morning"
    WHEN EXTRACT(HOUR FROM(time::time)) BETWEEN 12 AND 17 THEN "Afternoon"
    ELSE "Evening"
    END day_time,
COUNT(*)
FROM walmart_data
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

-- #9 Identify 5 branch with highest decrese ratio in revevenue compare to last year (current year 2023 and last year 2022)
WITH yearly_revenue AS (
    SELECT
        branch,
        SUM(CASE
            WHEN YEAR(TO_DATE(date, 'M/d/yyyy')) = 2022
            THEN unit_price * quantity
            ELSE 0
        END) AS revenue_2022,
        SUM(CASE
            WHEN YEAR(TO_DATE(date, 'M/d/yyyy')) = 2023
            THEN unit_price * quantity
            ELSE 0
        END) AS revenue_2023
    FROM walmart_data
    GROUP BY branch
)

SELECT
    branch,
    revenue_2022,
    revenue_2023,
    ROUND(
        (revenue_2022 - revenue_2023)
        / NULLIF(revenue_2022, 0) * 100,
        2
    ) AS decrease_percentage
FROM yearly_revenue
WHERE revenue_2022 > 0
  AND revenue_2023 < revenue_2022
ORDER BY decrease_percentage DESC
LIMIT 5;